In [ ]:
# ==============================
# 第一步：导入所需库
# ==============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# --- 建模相关的库 ---
# train_test_split: 把数据拆成「训练集」和「测试集」
#   训练集用来教模型学习规律，测试集用来考试看模型学得怎么样
#   如果不拆分，用同一批数据训练+考试，模型会"作弊"（过拟合），分数虚高
from sklearn.model_selection import train_test_split

# StandardScaler: 标准化（把不同量纲的特征拉到同一尺度）
#   比如价格范围 0~300，而小时数范围 0~23，不标准化的话
#   模型会误以为价格"更重要"（因为数值更大），标准化后系数才有可比性
from sklearn.preprocessing import StandardScaler

# LogisticRegression: 逻辑回归模型本身
#   虽然名字里有"回归"，但其实是做分类的（预测 0 或 1）
#   它输出的是一个 0~1 之间的概率值
from sklearn.linear_model import LogisticRegression

# 评估指标：
#   accuracy_score: 准确率 = 预测对的 / 总预测数
#   classification_report: 一张表汇总精确率、召回率、F1
#   roc_auc_score: ROC 曲线下面积，衡量模型区分正负样本的能力（0.5=瞎猜，1=完美）
#   roc_curve: 画 ROC 曲线用的
#   confusion_matrix: 混淆矩阵，看模型把哪些样本分对了/分错了
from sklearn.metrics import (accuracy_score, classification_report,
                             roc_auc_score, roc_curve, confusion_matrix)

# 中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# ==============================
# 第二步：加载数据
# ==============================
df = pd.read_csv(
    r'C:\Users\Administrator\Desktop\data_learn\eCommerce_Events_History\data\interim\03_user_behavior_groups.csv'
)

print(f'原始数据量：{len(df):,} 行, {len(df.columns)} 列')
print(f'\n分组分布：')
print(df['group_type'].value_counts())
print(f'\n缺失率：')
for col in ['brand', 'category_code', 'category_id', 'price']:
    print(f'  {col}: {df[col].isna().mean()*100:.1f}%')

In [ ]:
# ==============================
# 第三步：数据清洗
# ==============================

# --- 3.1 排除 B 组 ---
# B 组（被动流失）= 加购后既没买也没移出，结果不确定
# 逻辑回归需要明确的 Y：买了(1) vs 没买(0)
# B 组是"悬而未决"的状态，放进去会污染模型
df = df[df['group_type'].isin(['A', 'C'])].copy()
print(f'排除 B 组后剩余：{len(df):,} 行')
print(f'A 组（购买）: {(df["group_type"]=="A").sum():,} ({(df["group_type"]=="A").mean()*100:.1f}%)')
print(f'C 组（流失）: {(df["group_type"]=="C").sum():,} ({(df["group_type"]=="C").mean()*100:.1f}%)')

# --- 3.2 创建目标变量 Y ---
# A=1（买了）, C=0（主动移除了）
# 这就是模型要预测的目标：给一条记录，预测它属于 1 还是 0
df['y_purchased'] = (df['group_type'] == 'A').astype(int)
print(f'\n目标变量分布：购买={df["y_purchased"].sum():,}, 流失={len(df)-df["y_purchased"].sum():,}')

# --- 3.3 处理价格异常值 ---
# 之前发现 price 有负值（最低 -79.37），可能是退款数据
# 先看看有多少
neg_price = (df['price'] < 0).sum()
print(f'\n负价格记录：{neg_price} 条')
# 去掉负价格和零价格（零价格没意义，可能是赠品或错误数据）
df = df[df['price'] > 0].copy()
print(f'清洗后剩余：{len(df):,} 行')

# --- 3.4 对价格做对数变换 ---
# 价格分布通常是右偏的（大量低价 + 少量高价），取对数后更接近正态分布
# 好处：让模型对"相对变化"敏感（从 1 涨到 2 vs 从 100 涨到 200 同等重要）
df['log_price'] = np.log(df['price'])

In [ ]:
# ==============================
# 第三点五步：设计首次加购前 Session 特征
# ==============================
# 背景：
# 之前的 08_session_features.csv 是按“整段 session”聚合出来的画像表。
# 问题是：整段 session 会包含第一次加购之后、甚至移出购物车之后的行为。
# 如果直接把这些字段放进模型，模型可能会偷看答案，导致分数虚高。

# 本次重新设计字段的目标：
# 预测时点 = 每个 user_session 的第一次 cart 事件发生时。
# 只使用预测时点之前已经发生的行为，也就是：event_time < first_cart_time。
# 这样更接近真实业务场景：用户第一次加购时，我们预测他后续会不会购买。

# 字段工程伪代码模板：
# 1. 读取原始事件明细 df_raw，并确保 event_time 是 datetime 类型。
#    df_raw['event_time'] = pd.to_datetime(df_raw['event_time'])

# 2. 找到每个 session 的开始时间。
#    session_start_time = df_raw.groupby('user_session')['event_time'].min()
#    含义：这个 session 第一条事件发生的时间。

# 3. 找到每个 session 的首次加购时间。
#    first_cart_time = df_raw[df_raw['event_type'] == 'cart'].groupby('user_session')['event_time'].min()
#    含义：这个 session 第一次出现 cart 的时间，也就是本模型的预测时点。
#    注意：没有 cart 的 session 没有首次加购时点，暂时不进入这版特征表。

# 4. 把 first_cart_time 合并回原始事件明细。
#    目的：让每一条事件都知道自己所在 session 的首次加购时间。
#    合并后才能判断这一条事件是在首次加购前，还是首次加购后。

# 5. 筛选首次加购前事件。
#    df_pre_cart = df_raw_with_first_cart[df_raw_with_first_cart['event_time'] < df_raw_with_first_cart['first_cart_time']]
#    注意：这里用 <，不用 <=，因为第一次 cart 本身不能算作“加购前行为”。

# 6. 按 user_session 聚合 df_pre_cart，生成首次加购前的 session 特征。
#    这些字段描述的是用户在第一次加购之前的浏览/比较状态。

# 第一版计划生成的字段：
# - pre_cart_has_view：首次加购前是否有 view 行为，有则 1，没有则 0
# - pre_cart_view_count：首次加购前 view 事件次数
# - pre_cart_unique_products：首次加购前接触过的不同 product_id 数
# - pre_cart_unique_brands：首次加购前接触过的不同 brand 数
# - pre_cart_unique_categories：首次加购前接触过的不同 category_id 数
# - pre_cart_avg_price：首次加购前事件涉及商品的平均 price
# - pre_cart_max_price：首次加购前事件涉及商品的最高 price
# - minutes_to_first_cart：session_start_time 到 first_cart_time 的分钟数

# 特殊情况处理：
# 如果某个 session 第一条事件就是 cart，那么它在首次加购前没有任何 view。
# 这类 session 不应该删除，因为它可能代表目标明确、老用户回购或外部入口直达。
# 对这类 session：pre_cart_has_view = 0，浏览/品牌/品类/价格类 pre_cart 字段可填 0。

# 粒度提醒：
# 当前建模主表是 user_session × product_id 粒度。
# pre_cart 特征是 user_session 粒度。
# 合并回主表后，同一个 session 下的多个商品记录会共享同一组 pre_cart 特征。
# 因此解释模型时，pre_cart 字段只能解释“用户当次 session 的购物状态”，不能解释成某个商品本身的原因。